In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr


# -----------------------
# 0-1 Test Implementation
# -----------------------
def z1test(x, plot_pq=False, save_path=None, stock_name="", method="", level=""):
    x = np.array(x)
    if x.ndim > 1:
        x = x.flatten()
    N = len(x)
    j = np.arange(N)
    t = np.arange(1, int(N/10) + 1)
    c = np.pi/5 + np.random.rand(100) * 3*np.pi/5
    kcorr = np.zeros(100)
    p = np.zeros((N, 100))
    q = np.zeros((N, 100))

    for its in range(100):
        p[:, its] = np.cumsum(x * np.cos(j * c[its]))
        q[:, its] = np.cumsum(x * np.sin(j * c[its]))
        M = np.zeros(int(N/10))
        for n in range(int(N/10)):
            M[n] = np.mean((p[n+1:N, its] - p[:N-n-1, its])**2 +
                           (q[n+1:N, its] - q[:N-n-1, its])**2) - \
                   np.mean(x)**2 * (1 - np.cos((n+1)*c[its])) / (1 - np.cos(c[its]))
        kcorr[its], _ = pearsonr(t, M)

    k_val = np.median(kcorr)

    # Save plots if requested
    if plot_pq and save_path:
        skip = 33

        # p-q trajectories
        plt.figure(figsize=(8, 6))
        for i in range(5):
            plt.plot(p[skip:, i], q[skip:, i], linewidth=0.7, label=f'c[{i}]')
        plt.xlabel('p(n)')
        plt.ylabel('q(n)')
        plt.title(f'{stock_name} {method} L{level} - p-q Trajectories')
        plt.grid(True)
        plt.legend(fontsize='small')
        plt.tight_layout()
        plt.savefig(os.path.join(save_path, f"{stock_name}_{method}_L{level}_pq.png"))
        plt.close()

        # Time series
        plt.figure(figsize=(10, 4))
        plt.plot(x, 'r-', lw=0.6)
        plt.axvline(skip, color='gray', linestyle='--', label='Transient cutoff')
        plt.xlabel('Time')
        plt.ylabel('x(t)')
        plt.title(f'{stock_name} {method} L{level} - Time Series')
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(save_path, f"{stock_name}_{method}_L{level}_timeseries.png"))
        plt.close()

    return k_val


# -----------------------
# Batch Processing Script
# -----------------------
def process_clean_data(input_dir="clean_data",
                       summary_file="clean_data/denoising_summary.csv",
                       results_dir="0-1_results",
                       output_csv="0-1_results.csv"):

    if not os.path.exists(results_dir):
        os.makedirs(results_dir)

    # Load denoising summary so we know method + level for each stock
    summary_df = pd.read_csv(summary_file)
    summary_df.set_index("Ticker", inplace=True)

    results = []
    for file in os.listdir(input_dir):
        if not file.endswith("_clean.csv"):
            continue

        filepath = os.path.join(input_dir, file)
        df = pd.read_csv(filepath)

        # Extract ticker
        stock_name = file.replace("_clean.csv", "")

        # Lookup method and level from summary
        if stock_name not in summary_df.index:
            print(f"Warning: {stock_name} not found in summary table, skipping.")
            continue
        method = summary_df.loc[stock_name, "Best_Wavelet"]
        level = summary_df.loc[stock_name, "Best_Level"]

        # Get series
        if "Log_Return_Clean" not in df.columns:
            print(f"Warning: {file} missing Log_Return_Clean column, skipping.")
            continue
        data = df["Log_Return_Clean"].dropna().values
        sampled_data = data[::10]  # Subsample

        try:
            k_val = z1test(sampled_data, plot_pq=True, save_path=results_dir,
                           stock_name=stock_name, method=method, level=level)
            results.append([stock_name, method, level, k_val])
            print(f"Processed {stock_name} {method} L{level}: K = {k_val:.4f}")
        except Exception as e:
            print(f"Error processing {file}: {e}")

    # Save results as a table
    results_df = pd.DataFrame(results, columns=["Stock", "Method", "Level", "K"])
    results_df.to_csv(os.path.join(results_dir, output_csv), index=False)
    print(f"\nSaved results table to {os.path.join(results_dir, output_csv)}")


# -----------------------
# Run
# -----------------------
if __name__ == "__main__":
    process_clean_data()



Processed AMZN haar L2: K = 0.9965
Processed KLAC haar L2: K = 0.9974
Processed AAPL haar L2: K = 0.9978
Processed CSCO haar L2: K = 0.9974
Processed MSFT haar L2: K = 0.9988
Processed AMAT haar L2: K = 0.9979
Processed NFLX haar L2: K = 0.9975
Processed INTU haar L2: K = 0.9947
Processed GOOGL haar L2: K = 0.9977
Processed INTC haar L2: K = 0.9970
Processed NVDA haar L2: K = 0.9979
Processed TXN haar L2: K = 0.9983
Processed ADBE haar L2: K = 0.9979
Processed TSLA haar L2: K = 0.9972
Processed META haar L2: K = 0.9977
Processed QCOM haar L2: K = 0.9991
Processed MU haar L2: K = 0.9985
Processed LRCX haar L2: K = 0.9972

Saved results table to 0-1_results/0-1_results.csv
